# Module 2 — LLMs & Fine-Tuning: Topics & Practice

## Learning outcomes
1. Decide when fine-tuning is the right answer (and when prompting / RAG isn't).
2. Apply LoRA/QLoRA to a 7B model on consumer hardware.
3. Pick a quantization scheme for serving.
4. Build an instruction dataset that doesn't poison your model.

## 1. When NOT to fine-tune

Before you fine-tune, rule these out:
- **Prompting** is enough → keep it; cheaper to iterate.
- **RAG** solves the knowledge gap → no FT needed.
- **You have <500 high-quality examples** → FT will overfit; collect more.

Fine-tune when you need: a *style*, a *format*, a *latency budget* a small model can hit, or *private data* you can't ship to a hosted API.

> **Exercise 2.1** — For each scenario, choose: prompt / RAG / fine-tune / collect more data.
> a) Customer wants answers grounded in their internal wiki.
> b) Outputs must always be valid TOML.
> c) Replies must mimic a specific brand voice.
> d) The model should know events from yesterday.

## 2. Full FT vs PEFT vs LoRA vs QLoRA

| Approach | What it updates | VRAM (7B) | When |
|---|---|---|---|
| Full FT | every weight | ~80 GB | rarely needed |
| LoRA | small low-rank adapters | ~16 GB | the default |
| QLoRA | LoRA on a 4-bit quantized base | ~6 GB | consumer GPU |
| Prefix / P-tuning | virtual tokens | ~4 GB | very small tasks |

**LoRA hyper-params that matter:** `r` (rank, 8–64), `alpha` (≈ `2*r`), `target_modules` (attention `q_proj, v_proj` minimum; add `k_proj, o_proj, gate, up, down` for stronger fits), `dropout`.

> **Exercise 2.2** — Read the [LoRA paper abstract](https://arxiv.org/abs/2106.09685). In one paragraph, explain *why* a low-rank update preserves base-model knowledge.

## 3. Quantization for serving

After you train, you serve. Pick the smallest faithful representation.

| Format | Where | Loss | Notes |
|---|---|---|---|
| FP16/BF16 | training, GPU serving | none | baseline |
| INT8 (LLM.int8) | GPU | tiny | bitsandbytes |
| INT4 GPTQ / AWQ | GPU | small | calibration set required |
| GGUF Q4_K_M | CPU/Mac | small | llama.cpp / Ollama |

> **Exercise 2.3** — Given a 7B model on a 16 GB GPU, compute headroom for KV cache at FP16, INT8, and INT4. Which schemes leave room for a 4K context with batch=4?

## 4. Instruction tuning — dataset is everything

Format every example identically:
```
### Instruction:
{task}
### Input:
{optional_context}
### Response:
{answer}
```
Quality > quantity. **1k clean examples > 100k noisy ones.** Filter with: dedup, length bounds, refusal detection, LLM-as-judge for correctness.

> **Exercise 2.4** — Sketch a dataset pipeline that turns 50k raw support tickets into ~3k clean instruction examples. List the filters in order and what each removes.

## 5. RLHF / DPO — what it actually does

- Train a *reward model* on human preference rankings (chosen vs rejected).
- Optimise the LLM to maximise reward — PPO is the classic algorithm.
- **DPO** skips the explicit reward model; trains directly on preferences. Cheaper, works well.

You probably will not run RLHF at home. You *will* be a consumer of RLHF'd checkpoints — knowing the shape explains why the model refuses or sycophants.

> **Exercise 2.5** — Why does RLHF often *reduce* benchmark scores while *improving* user satisfaction? (One paragraph.)